# CV Based Rep Counter Tutorial
*Authors: Jonathan Holcombe, Spencer Merodio, Clarence Cheam, Helen Sun*


---
## Introduction
---

### Important Note

The webcam features demonstrated in this tutorial do not work in Google Colab due to issues with hardware-access permissions. Please download this tutorial as a Jupyter notebook to interact with these parts of the tutorial (`File>Download>Download .ipynb`).


---



### Background

When exercising in the gym, we can often lose track of how many reps we have done in a set, or forget to count our reps entirely. This tutorial aims to teach users how to use computer vision libraries like OpenCV and MediaPipe to assist with their gym workouts. Through machine learning (ML) models and computer vision (CV) techniques like pose estimation, users will build a framework to count their reps. Intended users for this tutorial are fitness lovers who are also interested in computer vision and AI. Users of this tutorial are expected to have some coding experience, preferably at the CS 5 level. No prior experience of using MediaPipe or OpenCV needed.

By completing this tutorial, users will be able to:

*  Learn how to use the Python computer vision libraries, specifically OpenCV.
*  Get experience with using Google's MediaPipe Pose Landmarker.
*  Combine these libraries to create a rep counting framework.

---

### Libraries in Focus

**`cv2`** - The Python OpenCV library, widely used for computer vision. In this tutorial, users will use this library to capture, convert, and generate images/videos.

**`numpy`** - Python library that is widely used for numerical operations, works especially well for arrays. Users will use numpy library alongside OpenCV to annotate inputs.

**`mediapipe`** - A multi-purpose ML model collection created by Google. In this tutorial, users will use mediapipe to analyze images and videos, detect and draw skeletons for body poses, then use the analyzed results to build the rep counter.

### Tutorial Outline

* Importing Packages
* OpenCV Color Conversion
* Body Pose Landmarker
  * Pose Landmarker Demonstration on Single Image
  * Creating a Pose Landmarker Object
  * Video Frame Annotation & Overlay
  * Connecting to Live Feed
* Rep Counter
  * Joint Angle Extraction
  * Rep Counting Shoulder Press
---

## Project Imports
Before we run any code, we need to make sure that the following packages are installed. By running this code cell below we are able to install them.

In [ ]:
!pip install numpy matplotlib opencv-python mediapipe gdown

Now import all required modules:

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils, drawing_styles

# Set Matplotlib settings to render axes without decorations
plt.rcParams.update({
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.bottom": False,
    "ytick.left": False,
    "xtick.labelbottom": False,
    "ytick.labelleft": False,
})

What each library does:
*   **`cv2`** - Used for capturing video, reading frames, and displaying annotated output
*   **`numpy`** - Used to store and manipulate image data as arrays
*   **`matplotlib`** - Used to display images in the notebook
*   **`mediapipe`** - Provides the models used for detecting body landmarks
*   **`mediapipe.tasks.python / vision`** - Provides the APIs used to create and run hand and pose landmark detectors
*   **`drawing_utils & drawing_styles`** - Used to draw landmarks and connections onto images

## cv2 Color Conversion

In all of the examples of pose estimator and rep counter below, we will be using `cv2` to process the video frames. However, OpenCV defaults to processing images in a blue-green-red (BGR) format, but MediaPipe models are trained on red-green-blue (RGB) images. This requires us to convert between these image formats, which we can do using the function `cv2.cvtColor()`.

First, download an example image (`lake.jpg`) from our Google drive:


In [ ]:
!gdown "1wgerLREKdtXiFns3h8YFFMiVCOKeVrLs"

To read an image using OpenCV, use the function `cv2.imread()` and pass the filename of the image you wish to display. Let's do this with `lake.jpg` in our example images. 

In [ ]:
lake_image = cv2.imread('lake.jpg')

This loads the image as a $\mathrm{height}\times \mathrm{width} \times 3$ NumPy array, where each of the three dimensions corresponds to a different color channel of the image. 

Let's visualize these channels! To display an image channel, we can use the `plt.imshow()` function  with a single channel to render it in the notebook. Here is a function that renders each channel separately:

In [ ]:
def display_image_channels(cv2_image):
    """Displays the three channels of a cv2 image alongside the combined image.

    The input image is a 3D NumPy array with shape (height, width, 3).

    Args:
        cv2_image (NDArray): Input cv2 image.

    Returns:
        None: Displays images using matplotlib.
    """
    _, axes = plt.subplots(1, 4, figsize=(12, 4)) # matplotlib axes to render the images on

    for idx, channel in enumerate(cv2.split(cv2_image)): # iterate over each channel
        # Create a valid image of just the selected channel
        zeros = np.zeros_like(channel)
        channels = [zeros, zeros, zeros]
        channels[idx] = channel
        channel_image = cv2.merge(channels)

        # Render the channel to the axis
        axes[idx].imshow(channel_image)
        axes[idx].set_title(f"Channel {idx}")

    # Render the combined image
    axes[3].imshow(cv2_image)
    axes[3].set_title("Combined")

    plt.show()

display_image_channels(lake_image)

The colors on this combined image seem off! Why is this?

When we read images or video frames using OpenCV, they are output in ***blue-green-red (BGR)*** format, as opposed to the typical ***red-green-blue (RGB)*** format. *(The reasons for this date back to the standards of cameras and software when OpenCV was being created.)* As such, the red and blue channels in our image are actually swapped! 

We need to convert the colorspace of this image. Converting between colorspaces can be done with the function `cv2.cvtColor(image, color_code)`, where:
* `image` is the OpenCV processed image output by `cv2.imread()`
* `color_code` defines the color conversion you want to complete with one of the codes in the [OpenCV ColorConversionCodes enumeration](https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html).


In [ ]:
# Convert the image from BGR to RGB
converted_image = cv2.cvtColor(lake_image, cv2.COLOR_BGR2RGB)
display_image_channels(converted_image)

That's more like it! Since the models that Google has trained in MediaPipe were all trained on images in the RGB format, they won't work if we give them images in BGR like this. We will be doing this for all of the video frames of our workouts!

---

## Setting Up the Pose Landmarker
Before you get into the details of the rep counter, you will first get experience with MediaPipe pose estimators and learn to set up a live pose landmarker. You will be able to detect live body movements and draw pose skeletons after completing this section.


MediaPipe landmarkers work by predicting the locations of a specific set of trained landmarks on the image (e.g., left wrist, right knee, etc.). These landmarks are each given a unique index, and every output of a landmarker has the landmarks ordered in accordance with this enumeration. Here are the landmarks that the PoseLandmarker outputs!

![](https://ai.google.dev/static/edge/mediapipe/images/solutions/pose_landmarks_index.png)

```
0 - nose
1 - left eye (inner)
2 - left eye
3 - left eye (outer)
4 - right eye (inner)
5 - right eye
6 - right eye (outer)
7 - left ear
8 - right ear
9 - mouth (left)
10 - mouth (right)
11 - left shoulder
12 - right shoulder
13 - left elbow
14 - right elbow
15 - left wrist
16 - right wrist
17 - left pinky
18 - right pinky
19 - left index
20 - right index
21 - left thumb
22 - right thumb
23 - left hip
24 - right hip
25 - left knee
26 - right knee
27 - left ankle
28 - right ankle
29 - left heel
30 - right heel
31 - left foot index
32 - right foot index
```

The pre-trained MediaPipe model weights are hosted on Google's storage API as `.task` files. Let's download the `.task` file for the pose estimator!

***Note***: if you find that the model is taking too long to detect the landmarkers each frame, you can try running a lighter version of the MediaPipe model. The weights for this model can be obtained by running the commented command.

In [ ]:
!curl -o pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task
# !curl -o pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker/float16/1/pose_landmarker.task # weights for a lighter model

To instantiate a MediaPipe landmarker, we need to first collect the "**options**" that the model will be run with. 

Every `.task` file has a set of base options included with it, and for our purposes we don't need to include any additional options. We can load these base options and assign them to be PoseLandmarker options with the following calls to the MediaPipe API:

In [ ]:
base_options = python.BaseOptions(model_asset_path="pose_landmarker.task")
options = vision.PoseLandmarkerOptions(base_options=base_options)

With these options collected, we can now create our PoseLandmarker.

In [ ]:
detector = vision.PoseLandmarker.create_from_options(options)

Let's load an image to test the landmarker with! Here is another example image from our Google drive (`yoga.png`).

In [ ]:
!gdown "1gw2ULhFbBQHue10A60_h70h5TNYmJJ5l"
yoga_image = cv2.cvtColor(cv2.imread('yoga.png'), cv2.COLOR_BGR2RGB)
plt.imshow(yoga_image);

In order to give this image to the PoseLandmarker, we need to first convert it to the image object that MediaPipe works with. `ImageFormat.SRGB` is the same as our previous RGB format, just in MediaPipe syntax.

In [ ]:
yoga_mp_image = mp.Image(data=yoga_image, image_format=mp.ImageFormat.SRGB)

Let's try detecting the pose in the image!

In [ ]:
detection_result = detector.detect(yoga_mp_image)
print(f"Pose Landmarks: {detection_result.pose_landmarks}")
print(f"Pose World Landmarks: {detection_result.pose_world_landmarks}")

This outputs two arrays of the 3D coordinates that the landmarker predicted the landmarks to be at: the `pose_landmarks` and `pose_world_landmarks`. These represent the same landmarks, but are in different coordinate systems:
* `pose_landmarks` is the normalized, ***image-space*** coordinate system, with $x$ and $y$ both ranging from $0$ to $1$. The $z$ coordinate is the relative depth of the landmarks relative to the image plane. This coordinate system is useful for rendering the landmarks back on the image.
* `pose_world_landmarks` is an unnormalized, ***world-space*** coordinate system, with the origin placed roughly around the hips. This coordinate system is useful for pose analysis and joint angle calculations.

Since MediaPipe doesn't provide a function to draw these landmarks back onto the original image, we need to define one ourselves:

In [ ]:
def draw_pose_landmarks_on_image(rgb_image, detection_result):
    """Draws the output pose landmarks in `detection_result` onto the `rgb_image`.

    The `rgb_image` is a 3D NumPy array representing a video frame with
    dimensions of `width` x `height` x `3` (one for each RGB channel).

    Args:
        rgb_image (NDArray): The 3D NumPy array representing the input image.
        detection_result (PoseLandmarkerResult): A result object from the PoseLandmarker.

    Returns:
        annotated_image (NDArray): A copy of `rgb_image` with the landmarks overlaid.
    """
    # The output image-space landmarks from the PoseLandmarker
    pose_landmarks_list = detection_result.pose_landmarks

    annotated_image = np.copy(rgb_image)

    # Styles for how the landmarks should be drawn
    pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
    pose_connection_style = drawing_utils.DrawingSpec(
        color = (0, 255, 0), # green connections
        thickness = 2
    )

    # Iterate through the landmarks to draw their nodes and connections
    for pose_landmarks in pose_landmarks_list:
        drawing_utils.draw_landmarks(
            image=annotated_image,
            landmark_list=pose_landmarks,
            connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
            landmark_drawing_spec=pose_landmark_style,
            connection_drawing_spec=pose_connection_style
        )

    return annotated_image

Let's try it out on our yoga image!

In [ ]:
yoga_annotated_image = draw_pose_landmarks_on_image(yoga_image, detection_result)
plt.imshow(yoga_annotated_image);

Success! We now have an estimate for the positions of each of the 33 landmarks.

However, if you look closely at the face, you may notice that:
1. The landmarks of the left eye are not displayed
2. The eye landmarks are a bit low of the person's actual eyes

This demonstrates two limitations of the pose estimation. First, if sections of the body are missing from the input image, their landmarks will be skipped, and secondly, landmark estimations may not be perfect. As such, the PoseLandmarker (and thus our rep counter) will perform better if we ensure that all relevant body parts are visible and as front-on as possible.

## Pose Landmarking with a Live Webcam Feed
This section will outline creating the core loop of our rep counter. In this loop, we will be:
- Reading a frame from a webcam
- Running the PoseLandmarker
- Drawing the landmarks to the frame
- Displaying the annotated frame back to the screen
You will use the `PoseLandmarker` object and the `draw_pose_landmarks_on_image` function created in the previous steps to get the landmarking working on a live webcam feed.


#### Step 1: Opening a Video Source with VideoCapture
To capture the frames from a video input device like a webcam, we can use `cv2.VideoCapture()`. This is a multi-purpose function whose behavior depends on the argument passed to it:
- If you pass it an integer, e.g. `cv2.VideoCapture(0)`, it will grab frames from the video input device whose index in your system devices corresponds to the integer passed. If you have more than one video input device, you can try changing the integer to correspond to the specific device you wish to use (`1`, `2`, etc.)
- If you pass it a filepath string that points to a video file, it grabs frames from that video file instead.


Either way, this will create a `VideoCapture` object, which can be iterated over to sequentially pull out frames from the video (we won't iterate right here, so there won't be any output yet). 

In [ ]:
cap = cv2.VideoCapture(0)

#### Step 2: Setting the Frame Dimensions
Webcams will usually default to their max resolution, so here we are explicitly telling OpenCV to request 640x480 pixels per frame. The two numbers ` 3 ` and `4` in the function call are **property IDs** and are shorthand for `cv2.CAP_PROP_FRAME_WIDTH` and `cv2.CAP_PROP_FRAME_HEIGHT`.

Smaller frames process much faster than larger frames, which is great for running the MediaPipe ML models in real-time. However, larger frames do give MediaPipe more detail to work with, so there is a balance to be struck here. 640x480 is a good starting spot for live pose estimation.

In [ ]:
cap.set(3, 640)  # width
cap.set(4, 480); # height

#### Step 3: Reading the Frames in a Loop
`cap.read()` gets the next frame from your video source and returns two things:
*   `success` a boolean variable, `True` if a frame was successfully captured, `False` otherwise.
*   `frame` an array of shape `(height, width, 3)`. Each pixel is represented by 3 numbers corresponding to the channels Blue, Green, Red (BGR).

We can pull out sequential frames with `cap.read()` in the following infinite loop. 

```
while True:
    success, frame = cap.read()
```

We will include the runnable code a bit later, since we don't have all the pieces we need yet. Importantly, we need a way to exit this loop. This can be done by detecting when a certain key is pressed, for example `q` for quit:

```
if cv2.waitKey(1) & 0xFF == ord("q"):
    cap.release()
    break
```

#### Step 4: Processing the Video Frames
Now we can apply the same processing and pose detection we did with the yoga image above:
- Convert the colorspace from BGR to RGB
- Package the array as an `mp.Image` object
- Run the PoseLandmarker on the processed frame
- Annotate the landmarks back onto the frame
- Convert the image back into a BGR `cv2` image
- Display the frame to the screen using `cv2.imshow`. Note that using `cv2.imshow` instead of `plt.imshow` displays the images in a separate window.

Putting all of these pieces together, we get the following loop. Try running it!

In [ ]:
while True:
    # Read a frame from the webcam
    success, frame = cap.read()

    # Convert the cv2 BGR frame to RGB
    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convert the frame to an SRGB MediaPipe Image object
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Run the PoseLandmarker
    detection_result = detector.detect(mp_image)

    # Draw the landmarks
    annotated_image = draw_pose_landmarks_on_image(
        mp_image.numpy_view(), detection_result
    )

    # Convert back to BGR for displaying with cv2
    output_frame = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imshow("Webcam Pose Landmarker", output_frame)

    # Break the loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        cap.release()
        break

# Clean up after exiting loop
cv2.destroyAllWindows()
cv2.waitKey(1);
 

If all went according to plan, you should be able to run the MediaPipe pose estimator on a live webcam feed and see the skeleton with the joints labeled in real time. If that's not what you see, try going back to step one and resetting the PoseLandmarker. This is essential for the rep counter we are going to build in later sections of the tutorial.

## Counting Reps

### Extracting Joint Angles
Now that we have estimates for the locations of each of the 33 landmarks output by the PoseLandmarker, we can do some math in order to calculate the angles of particular joints! It is these angles that we will be using to detect when a rep has occurred. Since we are doing calculations with these landmark locations, it is important that we use the world-space coordinates of the landmark, i.e. `detection_result.pose_world_landmarks`.

First, let's define an enum to make indexing the specific landmarks we want more understandable:

In [ ]:
from enum import Enum

class LANDMARKS(Enum):
    """MediaPipe pose landmark indices (0–32)."""
    NOSE = 0
    LEFT_EYE_INNER = 1
    LEFT_EYE = 2
    LEFT_EYE_OUTER = 3
    RIGHT_EYE_INNER = 4
    RIGHT_EYE = 5
    RIGHT_EYE_OUTER = 6
    LEFT_EAR = 7
    RIGHT_EAR = 8
    MOUTH_LEFT = 9
    MOUTH_RIGHT = 10
    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12
    LEFT_ELBOW = 13
    RIGHT_ELBOW = 14
    LEFT_WRIST = 15
    RIGHT_WRIST = 16
    LEFT_PINKY = 17
    RIGHT_PINKY = 18
    LEFT_INDEX = 19
    RIGHT_INDEX = 20
    LEFT_THUMB = 21
    RIGHT_THUMB = 22
    LEFT_HIP = 23
    RIGHT_HIP = 24
    LEFT_KNEE = 25
    RIGHT_KNEE = 26
    LEFT_ANKLE = 27
    RIGHT_ANKLE = 28
    LEFT_HEEL = 29
    RIGHT_HEEL = 30
    LEFT_FOOT_INDEX = 31
    RIGHT_FOOT_INDEX = 32

A joint (and its corresponding angle) is defined by three landmarks. Let's call them $a$, $b$, and $c$. We can use some linear algebra and trigonometry to calculate the angle formed by these three landmarks.

Let $\overrightarrow{ba}$ and $\overrightarrow{bc}$ be the displacement vectors from $b$ to $a$ and $b$ to $c$, respectively. Then the cosine of the angle $\theta$ between the two vectors is given by
$$
\cos \theta = \frac{\overrightarrow{ba} \cdot \overrightarrow{bc}}{\left|\overrightarrow{ba}\right|\left|\overrightarrow{bc}\right|}
$$
and we can take the inverse cosine of this to obtain our desired angle! Written as code, this looks like the following function:

In [ ]:
def calculate_angle(a, b, c):
    """Computes the angle in degrees between three `Landmark` objects.

    The input landmarks should be in world-space coordinates.

    Args:
        a, b, c (Landmark): Input world-space landmarks.

    Returns:
        theta (float): The angle at landmark `b` in degrees.
    
    """
    # Calculate displacement vectors
    ba = np.array([a.x - b.x, a.y - b.y, a.z - b.z])
    bc = np.array([c.x - b.x, c.y - b.y, c.z - b.z])

    # Compute the denominator
    mags = np.linalg.norm(ba) * np.linalg.norm(bc)
	
    if mags < 1e-6: # Avoid near-zero instability
        return 0.0

    # Compute the cosine and clamp to [-1, 1]
    cos_angle = np.dot(ba, bc) / mags
    cos_angle = np.clip(cos_angle, -1.0, 1.0)

    # Compute the arccos and convert to degrees
    theta = np.degrees(np.arccos(cos_angle))

    return theta 

We now need to pick a workout to count reps for! For this tutorial, we will be demonstrating counting reps for shoulder presses. Our implementation here will just consider the angle of the shoulders, but this could easily be extended to also consider other relevant angles like the elbow angles.

First let's define a function to calculate the shoulder angles given a detection result:

In [ ]:
def estimate_shoulder_angles(detection_result):
    """Estimates the angles formed by the landmarks creating the shoulder joints.

    Args:
        detection_result (PoseLandmarkerResult): A result object from the PoseLandmarker.

    Returns:
        angles (dict[str, float]): A dict containing estimates of the left and right shoulder angles.
    """
    if not detection_result.pose_world_landmarks:
        return {}

    landmarks = detection_result.pose_world_landmarks[0]  # assume one person

    L = LANDMARKS
    angles = {}

    angles['left_shoulder'] = calculate_angle(
        landmarks[L.LEFT_HIP],
        landmarks[L.LEFT_SHOULDER],
        landmarks[L.LEFT_ELBOW],
    )
    angles['right_shoulder'] = calculate_angle(
        landmarks[L.RIGHT_HIP],
        landmarks[L.RIGHT_SHOULDER],
        landmarks[L.RIGHT_ELBOW],
    )

    return angles


For the purpose of visualization, let's also define a function to write the estimated angles onto the frame as text:

In [ ]:
def display_angles_on_image(image, angles):
    """
    Display angle values as text on the input image
    
    Args:
        image (NDArray): The 3D NumPy array representing the input image.
        angles (dict[str, float]):  A dict containing the angles to be written.

    Returns:
        annotated_image (NDArray): A copy of `image` with the angle text overlaid.
    """
    # Text formatting
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    color = (0, 255, 0)
    thickness = 2

    # Y position of the current line (incremented)
    y_offset = 30

    annotated_image = np.copy(image)

    for joint_name, angle_value in angles.items():
        # Format joint name from dict key
        joint_name = joint_name.replace("_", " ").title() 

        # Create full text and get its dimensions
        text = f"{joint_name}: {round(angle_value)} deg"
        (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)

        # Place background rectangle and text on image
        cv2.rectangle(annotated_image, (0, y_offset-5), (text_w, y_offset + text_h + 5), (0,0,0), -1) 
        cv2.putText(annotated_image, text, (0, y_offset + text_h + font_scale - 1), font, font_scale, color, thickness)
        
        # Increment the Y position
        y_offset += 30

    return annotated_image

Before writing the object that actually counts the reps, let's try seeing if this shoulder angle extraction works! This loop is very similar to the previous landmarking loop, just with calls to these new functions we've written.

In [ ]:
cap = cv2.VideoCapture(0)
cap.set(3, 640)  # width
cap.set(4, 480)  # height

while True:
    # Read a frame from the webcam
    success, frame = cap.read()

    # Convert the cv2 BGR frame to RGB
    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convert the frame to an SRGB MediaPipe Image object
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Run the PoseLandmarker
    detection_result = detector.detect(mp_image)

    if not detection_result.pose_world_landmarks:
        continue

    # Draw the landmarks
    annotated_image = draw_pose_landmarks_on_image(
        mp_image.numpy_view(), detection_result
    )

    # Extract the shoulder angles and write them to the frame
    angles = estimate_shoulder_angles(detection_result)
    annotated_image = display_angles_on_image(annotated_image, angles)

    # Convert back to BGR for displaying with cv2
    output_frame = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imshow("Webcam Pose Landmarker", output_frame)

    # Break the loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        cap.release()
        break

# Clean up after exiting loop
cv2.destroyAllWindows()
cv2.waitKey(1);

As you may see in this, the angle estimations might not be the best. This is a fundamental limitation of trying to extract 3D information (landmark coordinates) from 2D data (the input image). MediaPipe does a decent job at determining the $z$ coordinate of the landmarks, but it still isn't perfect. 

What this means for us is that we will have to consider this uncertainty in the joint angles when setting the criteria for what counts as a "rep". In our implementation, we do this by defining a range of valid angles instead of a single angle. 

### The Rep Counter Object
With the angles now extracted, we are finally able to move onto counting the reps!

In order to count a rep, we need to keep track of two thresholds the joint angles need to cross, one threshold for when the rep is "down" and one for when it is "up". For a shoulder press, we might structure such a threshold like this:

In [ ]:
# NOTE: if you wish to add more exercises, include them here!
EXERCISE_LIST = {
    "shoulder_press":{
        "joint": "shoulder",
        "down_range": (40, 60),      # Arm down: 40-60 degrees
        "up_range": (140, 170),      # Arm up: 140-170 degrees
        "use_side": "right"          # Which shoulder to track
    },
}

Now, it's time for the actual RepCounter object. 

Here is an example of how you might go about implementing it. We've written this to run alongside the main webcam loop. We track two things: the rep count, and whether the user is in the "down" position. This latter bool is present so that we don't increment the rep count more than once before the "up" position is detected again.

In [ ]:
class RepCounter:
    """
    Tracks reps for a specified exercise using joint angle estimation.
    """
    
    def __init__(self, exercise_list, exercise_name):
        """
        Initialize the rep counter.
        
        Args:
            exercise_name (str): Name of the exercise to track
        """
        self._exercises = exercise_list
        self.exercise_name = exercise_name

        self.rep_count = 0
        self._is_in_down_position = True

        if exercise_name not in self._exercises:
            raise ValueError(f"Exercise '{exercise_name}' not supported")

        self._exercise = self._exercises[exercise_name]
    
    def update(self, angles):
        """
        Update the rep counter using the latest joint angles.

        Args:
            angles (dict): Dictionary of joint angles.

        Returns:
            None
        """
        if not angles:
            return 0
        
        # Get the appropriate shoulder angle
        joint_key = f"{self._exercise['use_side']}_{self._exercise['joint']}"
        
        if joint_key not in angles:
            return 0
        
        current_angle = angles[joint_key]
        down_min, down_max = self._exercise["down_range"]
        up_min, up_max = self._exercise["up_range"]
        
        
        # Currently down -> look for up
        if self._is_in_down_position:
            if up_min <= current_angle <= up_max:
                self._is_in_down_position = False

        # Currently up -> look for down
        else:
            if down_min <= current_angle <= down_max:
                self._is_in_down_position = True
                self.rep_count += 1
    
    def get_rep_count(self):
        return self.rep_count
    
    def reset(self):
        self.rep_count = 0
        self._is_in_down_position = True

Let's now initialize our RepCounter and plug it into our main webcam loop. If all goes well, this should be counting shoulder press reps! Depending on your particular camera angle, lighting, etc., you may have to fine tune the angle thresholds.

In [ ]:
rep_counter = RepCounter(exercise_list=EXERCISE_LIST, exercise_name="shoulder_press")

cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

while True:
    # Read a frame from the webcam
    success, frame = cap.read()

    # Convert the cv2 BGR frame to RGB
    rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convert the frame to an SRGB MediaPipe Image object
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

    # Run the PoseLandmarker
    detection_result = detector.detect(mp_image)

    if not detection_result.pose_world_landmarks:
        continue

    # Draw the landmarks
    annotated_image = draw_pose_landmarks_on_image(
        mp_image.numpy_view(), detection_result
    )

    # Extract the shoulder angles and write them to the frame
    angles = estimate_shoulder_angles(detection_result)
    annotated_image = display_angles_on_image(annotated_image, angles)

    # Update the RepCounter and write rep count text to frame
    rep_counter.update(angles)
    rep_text = f"Reps: {rep_counter.get_rep_count()}"
    cv2.putText(annotated_image, rep_text, (400, 400), cv2.FONT_HERSHEY_SIMPLEX, 
                1.5, (0, 255, 255), 3)

    # Convert back to BGR for displaying with cv2
    output_frame = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
    cv2.imshow("Webcam Pose Landmarker", output_frame)

    # Break the loop when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord("q"):
        cap.release()
        break

# Clean up after exiting loop
cv2.destroyAllWindows()
cv2.waitKey(1);

Congrats, you just officially completed the tutorial at this point! We learned how to use OpenCV and MediaPipe to work with video frames and set up the pose landmarker, and we also derived a formula and explored ways to estimate joint angles. At the end, we built a rep counter for counting shoulder presses together. Now you should not only be familiar with image and video processing in OpenCV, but also be well equipped with all the techniques you need to build your own rep counter. Using the same algorithm, you can select a few exercises you like and code out the threshold angles for counting reps just like what we did together for shoulder press. We encourage you to build on the framework we provide in this tutorial, making the rep counter specific to your workout plans and finding a way to use the rep counter that best assists your workout! 